# 2단계 · 진짜 오픈모델 파인튜닝 (Qwen + LoRA)

이제 실제로 쓸 수 있는 **한국어 Q&A 챗봇**을 만듭니다.
`Qwen/Qwen2.5-0.5B-Instruct` 모델을 다운로드해서, 내 Q&A 데이터로 **LoRA 파인튜닝**합니다.

- **왜 Qwen 0.5B?** 한국어를 잘하고, T4 무료 GPU에서도 LoRA로 학습 가능한 작은 크기라서.
- **왜 LoRA?** 전체 모델 대신 작은 추가 가중치만 학습 → 적은 메모리로 빠르게.

> 실행 전: **런타임 → 런타임 유형 변경 → T4 GPU** 필수!


## 0. 라이브러리 설치
파인튜닝에 필요한 HuggingFace 생태계 도구들입니다.


In [ ]:
# torchao는 LoRA fp16 학습에 불필요한데, Colab 기본 버전(0.10)이 최신 peft와 충돌하므로 먼저 제거
!pip uninstall -q -y torchao
!pip install -q -U transformers datasets peft trl accelerate bitsandbytes
import torch, transformers, trl
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU(!)')

## 1. 학습 데이터 준비

`question`/`answer` 쌍이 담긴 JSONL이 필요합니다. 두 가지 방법:

**(A) 직접 만든 데이터 업로드** — 이 저장소의 `train/data/sample_qa.jsonl`을 Colab에 업로드.
**(B) 아래 셀이 샘플 데이터를 자동 생성** (업로드가 번거로우면 이걸 쓰세요).

실제 챗봇 품질은 **데이터 양과 질**이 좌우합니다. 최소 수백 개를 권장하지만, 여기선 흐름을 익히는 게 목적입니다.


In [ ]:
# (B) 샘플 데이터 자동 생성 — 직접 데이터가 있으면 이 셀을 건너뛰고 sample_qa.jsonl을 업로드하세요
import json, os
samples = [
  {'question':'벡터 데이터베이스가 뭐예요?','answer':'벡터 데이터베이스는 텍스트나 이미지를 숫자 벡터(임베딩)로 바꿔 저장하고, 의미가 비슷한 항목을 빠르게 찾아주는 데이터베이스입니다.'},
  {'question':'임베딩이 뭔가요?','answer':'임베딩은 단어나 문장을 고정 길이의 숫자 벡터로 바꾼 표현으로, 의미가 비슷하면 벡터 공간에서 가까이 위치합니다.'},
  {'question':'RAG가 무슨 뜻이에요?','answer':'RAG는 검색 증강 생성의 약자로, 질문과 관련된 문서를 먼저 벡터 DB에서 찾은 뒤 그 내용을 참고 자료로 LLM에 넣어 답을 생성하는 방식입니다.'},
  {'question':'LoRA는 왜 쓰나요?','answer':'LoRA는 거대한 모델 전체 대신 작은 추가 행렬만 학습하는 기법으로, 적은 GPU 메모리로 효율적으로 파인튜닝할 수 있습니다.'},
  {'question':'파인튜닝이 뭐예요?','answer':'파인튜닝은 이미 학습된 큰 모델을 내가 원하는 데이터로 추가 학습시켜 말투나 지식을 조정하는 과정입니다.'},
]
# 학습 효과를 위해 반복 증강 (실전에선 진짜 다양한 데이터를 모으세요)
with open('sample_qa.jsonl','w',encoding='utf-8') as f:
    for _ in range(40):
        for s in samples:
            f.write(json.dumps(s, ensure_ascii=False)+'\n')
print('생성 완료:', os.path.getsize('sample_qa.jsonl'), 'bytes')

## 2. 데이터셋 로드 & 채팅 형식으로 변환

Instruct 모델은 '채팅 템플릿' 형식으로 학습해야 합니다.
각 Q&A를 `시스템 → 사용자 질문 → 모델 답변` 대화로 바꿉니다.


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

SYSTEM = '당신은 벡터DB, 임베딩, RAG, LLM을 쉽고 친절하게 설명하는 한국어 도우미입니다.'

ds = load_dataset('json', data_files='sample_qa.jsonl', split='train')

def to_chat(ex):
    msgs = [
        {'role':'system','content': SYSTEM},
        {'role':'user','content': ex['question']},
        {'role':'assistant','content': ex['answer']},
    ]
    ex['text'] = tokenizer.apply_chat_template(msgs, tokenize=False)
    return ex

ds = ds.map(to_chat)
print(ds[0]['text'])

## 3. 모델 로드 (+ LoRA 설정)

메모리를 아끼려고 모델을 줄여 불러오고, LoRA 어댑터를 붙입니다.
`trainable params`가 전체의 1% 미만인 걸 확인하세요 — 이게 LoRA의 핵심입니다.


In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.config.use_cache = False

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 4. 파인튜닝 실행 (SFT)

SFT(지도 미세조정) = '질문 → 좋은 답변' 쌍을 보여주며 따라 하게 가르치기.
loss가 내려가면 학습이 잘 되는 것입니다. 샘플 데이터 기준 T4에서 몇 분이면 끝납니다.


In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir='qwen-ko-qa-lora',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy='epoch',
    bf16=False, fp16=True,
    dataset_text_field='text',
    max_length=512,   # 최신 TRL에서 max_seq_length → max_length 로 이름이 바뀜
    report_to='none',
)

trainer = SFTTrainer(model=model, train_dataset=ds, args=cfg)
trainer.train()

## 5. 추론 테스트 — 내 챗봇과 대화하기

학습한 모델에게 질문해 봅니다. 데이터에 있던 주제는 우리가 가르친 말투로 답할 거예요.


In [ ]:
def ask(question, max_new_tokens=256):
    msgs = [{'role':'system','content':SYSTEM}, {'role':'user','content':question}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.7, top_p=0.9)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return text

print(ask('벡터 데이터베이스가 뭐예요?'))
print('---')
print(ask('RAG와 파인튜닝의 차이를 알려줘'))

## 6. 모델 저장 — 두 가지 방법

**(A) LoRA 어댑터만 저장** — 작고(수 MB) 가벼움. 쓸 때 원본 모델과 함께 로드.
**(B) 병합(merge)** — LoRA를 원본에 합쳐 독립적인 완성 모델로 만듦. 서빙에 편리.


In [ ]:
# (A) 어댑터만 저장
trainer.save_model('qwen-ko-qa-lora')
tokenizer.save_pretrained('qwen-ko-qa-lora')
print('어댑터 저장 완료')

In [ ]:
# (B) 원본에 병합해 완성 모델 만들기 (서빙용)
# 방금 학습해 '메모리에 있는' model을 바로 병합 → 디스크에서 다시 로드하지 않아 안전합니다.
merged = model.merge_and_unload()
merged.save_pretrained('qwen-ko-qa-merged')
tokenizer.save_pretrained('qwen-ko-qa-merged')
print('병합 모델 저장 완료 → qwen-ko-qa-merged')

# 참고: 나중에 '저장된 어댑터'로부터 병합하려면 반드시 로컬 경로임을 './'로 명시하세요.
#   from peft import PeftModel
#   from transformers import AutoModelForCausalLM
#   base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')
#   merged = PeftModel.from_pretrained(base, './qwen-ko-qa-lora').merge_and_unload()  # './' 필수
# './' 없이 'qwen-ko-qa-lora'만 주면 PEFT가 Hub 저장소로 오해해 401 에러가 납니다.

## 6-2. GGUF로 변환 — 로컬 CPU(Ollama)에서 돌리기 위한 단계

내 PC에 NVIDIA GPU가 없으면, 병합 모델을 **GGUF 포맷**으로 바꿔야 로컬 CPU에서 Ollama로 돌릴 수 있습니다.
아래 셀이 `qwen-ko-qa-merged` 폴더를 `qwen-ko-qa.gguf` 한 파일로 변환하고 내 PC로 다운로드합니다.

- `--outtype q8_0`: 8비트 양자화. 0.5B 기준 약 0.5GB로 작고, CPU에서 빠르고 품질 손실도 적습니다.
  - 더 작게: `q4_k_m` (약 0.4GB, 약간 더 가벼움). 품질 우선이면 `f16`.
- 변환에는 `## 6`의 (B) 병합 셀을 먼저 실행해 `qwen-ko-qa-merged` 폴더가 있어야 합니다.

> 다운로드한 `qwen-ko-qa.gguf`는 로컬에서 Ollama에 등록합니다 (방법은 마지막 '3단계' 셀 참고).


In [ ]:
# 병합 모델(qwen-ko-qa-merged) → GGUF 변환 후 내 PC로 다운로드
import os
assert os.path.isdir('qwen-ko-qa-merged'), '먼저 ## 6 (B) 병합 셀을 실행하세요!'

# llama.cpp 변환 스크립트 준비
if not os.path.isdir('llama.cpp'):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt

# 변환 (q8_0 = 8비트 양자화, 0.5B 기준 약 0.5GB)
!python llama.cpp/convert_hf_to_gguf.py qwen-ko-qa-merged \
    --outfile qwen-ko-qa.gguf --outtype q8_0

print('GGUF 크기:', round(os.path.getsize('qwen-ko-qa.gguf')/1e6, 1), 'MB')

# 내 PC로 다운로드 (브라우저 다운로드 창이 뜹니다)
from google.colab import files
files.download('qwen-ko-qa.gguf')

## 7. (선택) HuggingFace Hub에 올리기

Colab 런타임은 종료되면 파일이 사라집니다. 모델을 보관/공유하려면 Hub에 푸시하세요.
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)에서 write 토큰을 발급받으세요.


In [ ]:
## 3단계 · 이 VectorDB 프로젝트에 연결하기

핵심: 이 앱(`app.py`)이 벡터DB 검색은 이미 처리하고, **LLM 호출만 OpenAI 호환 주소로** 보냅니다.
따라서 내 모델을 그 주소로 띄우고 `.env`의 `VLLM_*` 3개만 바꾸면 코드 수정 없이 연결됩니다.

### 방법 1) Ollama로 로컬 CPU 서빙 (GPU 없을 때 — 권장)

`6-2` 셀에서 받은 `qwen-ko-qa.gguf`를 내 PC에서:

1. [ollama.com](https://ollama.com)에서 Ollama 설치 (설치 시 서버가 자동 실행됨)
2. gguf 파일 옆에 `Modelfile` 생성 (확장자 없음):
   ```
   FROM ./qwen-ko-qa.gguf
   ```
3. PowerShell:
   ```powershell
   ollama create qwen-ko-qa -f Modelfile
   ollama run qwen-ko-qa "벡터 데이터베이스가 뭐야?"   # 답하면 성공
   ```
4. 이 프로젝트 `.env` 수정:
   ```dotenv
   VLLM_URL=http://localhost:11434/v1
   VLLM_API_KEY=ollama
   VLLM_MODEL=qwen-ko-qa
   ```
5. 실행: `streamlit run app.py`

### 방법 2) GPU 서버/Colab에서 vLLM 서빙 (Hub에 올린 경우)
```bash
pip install vllm
python -m vllm.entrypoints.openai.api_server --model 내아이디/qwen-ko-qa --port 8000
```
```dotenv
VLLM_URL=http://<서버주소>:8000/v1
VLLM_API_KEY=dummy
VLLM_MODEL=내아이디/qwen-ko-qa
```

### 정리
`from scratch`로 원리를 이해하고 → 진짜 모델을 LoRA로 파인튜닝하고 →
GGUF로 바꿔 로컬에서 서빙하고 → 내 RAG 앱에 붙이는 전체 흐름을 완성했습니다.
다음은 **데이터(`sample_qa.jsonl`)를 늘려** 품질을 올리는 단계입니다. 🚀


## 3단계 · 이 VectorDB 프로젝트에 연결하기

이 저장소는 OpenAI 호환 LLM 엔드포인트를 사용합니다(`.env`의 `VLLM_URL`/`VLLM_MODEL`).
방금 만든 모델을 **vLLM으로 서빙**하면 그대로 연결됩니다.

### 방법 1) GPU 서버/Colab에서 vLLM로 서빙
```bash
pip install vllm
# Hub에 올린 경우
python -m vllm.entrypoints.openai.api_server \
    --model 내아이디/qwen-ko-qa --port 8000
```
그러면 이 프로젝트의 `.env`를 이렇게 바꿉니다:
```dotenv
VLLM_URL=http://<서버주소>:8000/v1
VLLM_API_KEY=dummy
VLLM_MODEL=내아이디/qwen-ko-qa
```

### 방법 2) GPU가 없다면 — Ollama로 로컬 CPU 서빙 (가장 현실적)
병합 모델을 GGUF로 변환하면 CPU에서도 돌릴 수 있습니다.
`llama.cpp`의 `convert_hf_to_gguf.py`로 변환 후 Ollama에 등록하면,
Ollama가 OpenAI 호환 엔드포인트(`http://localhost:11434/v1`)를 제공하므로 동일하게 연결됩니다.

### 정리
`from scratch`로 원리를 이해하고 → 진짜 모델을 LoRA로 파인튜닝하고 →
내 RAG 앱에 붙이는 전체 흐름을 완성했습니다. 다음은 **데이터를 늘려** 품질을 올리는 단계입니다. 🚀
